# Tutorial 1

> Tutorial 1

In [1]:
#| default_exp tutorial_1

In [2]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import download_medmnist

from monai.utils import set_determinism
from monai.transforms import ScaleIntensity

set_determinism(0)

In [3]:
device = get_device()
print(device)

cuda


### Download Data

In [28]:
image_path = '../_data/medmnist_data/'
download_medmnist('pathmnist', image_path, download_only=True)

100%|██████████| 205615438/205615438 [00:20<00:00, 10269948.83it/s]


Using downloaded and verified file: ../_data/medmnist_data/pathmnist.npz
Using downloaded and verified file: ../_data/medmnist_data/pathmnist.npz
Saving training images to ../_data/medmnist_data/...


100%|██████████| 89996/89996 [00:24<00:00, 3703.44it/s]


Saving validation images to ../_data/medmnist_data/...


100%|██████████| 10004/10004 [00:02<00:00, 3733.77it/s]


Saving test images to ../_data/medmnist_data/...


100%|██████████| 7180/7180 [00:01<00:00, 3709.81it/s]

Removed pathmnist.npz
Datasets downloaded to ../_data/medmnist_data/


### Create Dataloader

In [ ]:
bs, size = 16, 128

path = Path('../_data/Babesia/')
path_x = path/'RI'
path_y = path/'TRITC'


data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageProject), BioImageBlock(cls=BioImage)),
    'get_items':    get_image_files,
    'get_y':        get_target(path_y, same_filename=False, signal_file_prefix='RI', target_file_prefix='TRITC'),
    'splitter':     RandomSplitter(valid_pct=0.2),
    'item_tfms':    [ScaleIntensity(minv=0.0, maxv=1.0),RandCrop2D(size), RandRot90(prob=0.5), RandFlip(prob=0.75)],
    'bs': bs,
}

data = get_dataloader(
    path_x, 
    show_summary=False,
    **data_ops,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
data.show_batch(max_n=2, cmap='hot')

### Load and train a 2D model

In [ ]:
from bioMONAI.nets import Deeplab, DeeplabConfig

In [ ]:
config_2d = DeeplabConfig(
    dimensions=2,
    in_channels=1,  
    out_channels=1,
    backbone="resnet10",  
    aspp_dilations=[1, 6, 12, 18]
)
model = Deeplab(config_2d)
 
loss = MSSSIML1Loss(2, levels=2) #CombinedLoss(alpha=0, beta=0.5)
metrics = [SSIMMetric, MSELoss]

trainer = fastTrainer(data, model, loss_fn=loss, metrics=metrics, show_summary=False)

In [ ]:
trainer.fit_flat_cos(500)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!